In [1]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import warnings

warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


In [2]:
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
for sub in ["data/raw", "data/processed", "data/simulated",
            "notebooks", "src", "models", "logs", "docs"]:
    (ROOT / sub).mkdir(parents=True, exist_ok=True)

print("Racine du projet :", ROOT)
print(sorted(p.name for p in ROOT.iterdir()))

Racine du projet : C:\Users\USER\anaconda_projects\ae0031cb-ad1c-4e8e-b428-a8bce55a9655
['.ipynb_checkpoints', 'data', 'docs', 'logs', 'models', 'notebooks', 'src']


In [3]:
!pip install kagglehub --quiet

In [4]:
#upload dataset
import kagglehub, shutil

cache_path = Path(kagglehub.dataset_download(
    "arnabbiswas1/microsoft-azure-predictive-maintenance"))
print("Cache :", cache_path)

for f in cache_path.rglob("*.csv"):
    shutil.copy(f, ROOT / "data" / "raw" / f.name)
    print("copié :", f.name)

Cache : C:\Users\USER\.cache\kagglehub\datasets\arnabbiswas1\microsoft-azure-predictive-maintenance\versions\3
copié : PdM_errors.csv
copié : PdM_failures.csv
copié : PdM_machines.csv
copié : PdM_maint.csv
copié : PdM_telemetry.csv


In [5]:
#vérification des fichiers
expected = ["PdM_telemetry.csv", "PdM_errors.csv", "PdM_maint.csv",
            "PdM_failures.csv", "PdM_machines.csv"]
raw = ROOT / "data" / "raw"
present = {p.name for p in raw.glob("*.csv")}

for name in expected:
    p = raw / name
    if name in present:
        with open(p, "rb") as fh:
            n_lines = sum(1 for _ in fh) - 1   # sans l'en-tête
        print(f"OK   {name:20s} {p.stat().st_size/1e6:7.2f} Mo   {n_lines:>8,} lignes")
    else:
        print(f"MANQUANT  {name}")

print("\nFichiers présents :", sorted(present))

OK   PdM_telemetry.csv      80.14 Mo    876,100 lignes
OK   PdM_errors.csv          0.13 Mo      3,919 lignes
OK   PdM_maint.csv           0.10 Mo      3,286 lignes
OK   PdM_failures.csv        0.02 Mo        761 lignes
OK   PdM_machines.csv        0.00 Mo        100 lignes

Fichiers présents : ['PdM_errors.csv', 'PdM_failures.csv', 'PdM_machines.csv', 'PdM_maint.csv', 'PdM_telemetry.csv']


In [6]:
# fichier de configuration partagé
config_code = '''
from pathlib import Path

ROOT = Path(__file__).resolve().parent.parent
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
SIMULATED = ROOT / "data" / "simulated"
MODELS = ROOT / "models"
LOGS = ROOT / "logs"

SEED = 42
START = "2015-01-01 06:00:00"
END = "2016-01-01 06:00:00"
N_MACHINES = 100
'''
(ROOT / "src" / "config.py").write_text(config_code.strip() + "\n", encoding="utf-8")


312

In [7]:
#testd'import de la configuration
sys.path.append(str(ROOT / "src"))
from config import RAW, SEED
print("SEED =", SEED)
print("RAW existe :", RAW.exists())

SEED = 42
RAW existe : True
